<div style="background-color: lightblue; color: black; border: 2px solid black; 
padding: 15px 25px; border-radius: 8px; margin: 10px 0px;">

  <h2 style="color: black; margin: 0px 0px 10px 0px;">
   Rain gauge optimisation
  </h2>

  <hr style="border: 1px solid black; margin: 10px 0px;">

  <p>This code works on defining optimal locations for placements of new rain gauges in the Upper Severn catchment. The placement of the gauges aims to optimise the accuracy with which rainfall over the catchment can be predicted. </p>

  <p><strong>The following data is available:</strong></p>
  <ul>
    <li> 1km HAD-UK gridded annual (?) rainfall data </li>
    <li> Rain gauge data from 10? existing rain gauges</li>
    <li> 30m (?) DEM data: from which slope and aspect can be derived. These are then aggregated to 1km </li>      
  </ul>
<!-- 
  <div style="background-color: white; border-left: 4px solid orange; 
  padding: 8px 15px; border-radius: 0px 5px 5px 0px; margin-top: 12px;">
    <strong>NB:</strong> The threshold for an "unphysical" value is 
    currently set at <strong>150mm/hr</strong> — selected relatively 
    arbitrarily and may need review.
  </div> -->

</div>

In [1]:
import pandas as pd
import geopandas as gpd
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import folium
import pickle
from shapely.ops import unary_union
import rioxarray
import xarray as xr
from pathlib import Path
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import box

data_dir = "/scratch/hydro4/users/kv25483/FDRI/Rain_gauge_optimisation/Data/"

In [2]:
# Load Plynlimon catchments shapefile
Plyn_catch = gpd.read_file(data_dir + "Headwater_catchments/Plyn_catchments/data/Shapefile/"
    "ee2eeee5-e456-4e93-87e9-eee97ee149ee/PlynlimonCatchments.shp").set_crs(epsg=27700, allow_override=True)

# Load Plynlimon river network
Plyn_RN = gpd.read_file(data_dir + "Headwater_catchments/Plyn_rivernet/data/Shapefile/PlynlimonRiverNetwork.shp").set_crs(
    epsg=27700, allow_override=True)

# Load 2023 Land Cover data from CEH (10m res)
catchment_LMC = rioxarray.open_rasterio(data_dir + "lmc2023 clip.tif")

# Load DEMs
dem_Dolwen_1m = rioxarray.open_rasterio(data_dir + "dem_Dolwen_1m.tif")
dem_Dolwen    = rioxarray.open_rasterio(data_dir + "dem_Dolwen.tif")
dem_wider     = rioxarray.open_rasterio(data_dir +  "dem_Dolwen_wider.tif")

# Load other shapefiles
catchment_Dolwen = gpd.read_file(data_dir + "54080/54080.shp")
river_net_wider = gpd.read_file(data_dir + "54080_river_net/StreamLine_54080.shp").set_crs(epsg=27700)
river_net_Dolwen = gpd.read_file(data_dir + "river_net_Dolwen.shp")
river_net_CEH = gpd.read_file(data_dir + "River_net/DigitalRiverNetwork_50K_GB.shp").set_crs(epsg=27700)

# ==============================================================
# Load Soil Info
# ==============================================================
BGS_soil_shp = gpd.read_file(data_dir + "BGS_soil_parent.shp")

# Load and reproject HOST raster to EPSG:27700
BGS_HOST = rioxarray.open_rasterio(data_dir + "HOST.tif")
BGS_HOST = BGS_HOST.rio.reproject("EPSG:27700")

# Clip soil polygon to Dolwen catchment
BGS_soil_Dolwen = gpd.overlay(BGS_soil_shp, catchment_Dolwen, how="intersection")

# Mask HOST raster to Dolwen catchment
BGS_HOST_Dolwen = BGS_HOST.rio.clip(catchment_Dolwen.geometry, crs=catchment_Dolwen.crs, drop=True)

# ==============================================================
# Spatial Intersections
# ==============================================================

# Clip Plynlimon river network to Dolwen catchment
Plyn_RN_S = gpd.overlay(Plyn_RN, catchment_Dolwen, how="intersection")

# Filter Plynlimon catchments to Severn only
Plyn_catch_S = Plyn_catch[Plyn_catch["Catchment"] == "Severn"]

# Intersect Dolwen river network with Severn Plynlimon catchment
Plyn_RN_S_10k = gpd.overlay(river_net_Dolwen, Plyn_catch_S, how="intersection")

# ==============================================================
# Load Gauge and Weather Station Locations
# ==============================================================
plyn_sites = pd.read_csv(data_dir+ "Headwater_catchments/Plyn_sites_meta.csv")

# Convert to GeoDataFrame using BNG coordinates
plyn_sites_bng = gpd.GeoDataFrame(plyn_sites, geometry=gpd.points_from_xy(plyn_sites["x"], plyn_sites["y"]), crs="EPSG:27700")

### Get rain gauge data

In [3]:
# Load rain gauge metadata
rain_meta_cmd = pd.read_csv(data_dir + "rain_meta_for_CMD.csv")

# Convert to GeoDataFrame (BNG coordinates)
rain_meta_sf = gpd.GeoDataFrame(rain_meta_cmd, geometry=gpd.points_from_xy(rain_meta_cmd["x_2"], rain_meta_cmd["y_2"]),
                                crs="EPSG:27700")

# CRS definitions (for reference)
ukgrid  = "EPSG:27700"
latlong = "EPSG:4326"

### Get gridded rainfall data

In [4]:
HAD_names = sorted(Path(data_dir + "HAD_rainfall_annual/").glob("*.nc"))
HAD_layers = [xr.open_dataset(f) for f in HAD_names]
HAD_ann = xr.concat([ds['rainfall'] for ds in HAD_layers], dim="time")

HAD_ann = HAD_ann.rio.write_crs("EPSG:27700")
HAD_Dolwen_ppn = HAD_ann.rio.clip(catchment_Dolwen.geometry, crs=catchment_Dolwen.crs,drop=True, all_touched=True)
HAD_Dolwen_ppn_mean = HAD_Dolwen_ppn.mean(dim="time")

### Get terrain data (DEM and then derive slope and aspect)

In [5]:
# Define location of DEM
dem_path = data_dir + "dem_Dolwen.tif"

# 1. Load DEM
with rioxarray.open_rasterio(dem_path, masked=True) as dem_rx:
    dem = dem_rx.squeeze()
    dem_arr = dem.values.astype(float)
    transform = dem.rio.transform()

# 2. Get grid resolution    
dx = transform.a   # pixel width
dy = -transform.e  # pixel height (positive)    

# 3. Compute slope + aspect (drop-in replacement)
dz_dy, dz_dx = np.gradient(dem_arr, dy, dx)

# Slope (degrees)
slope_arr = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))

# Aspect (degrees, 0–360)
aspect_arr = np.degrees(np.arctan2(-dz_dx, dz_dy))
aspect_arr = np.where(aspect_arr < 0, 360 + aspect_arr, aspect_arr)

#from scipy.ndimage import generic_filter

# def tri_func(window):
#     center = window[len(window)//2]
#     return np.mean(np.abs(window - center))

# tri_arr = generic_filter(dem_arr, tri_func, size=3)

### Define a 1km grid based on the rainfall data (to use for aggregating terrain data)

In [6]:
# Get coordinates
x = HAD_ann.projection_y_coordinate.values
y = HAD_ann.projection_y_coordinate.values

# Get resolution (assumes regular grid)
dx = np.abs(x[1] - x[0])
dy = np.abs(y[1] - y[0])

# Build polygons
polys = []
for xi in x:
    for yi in y:
        polys.append(box(xi - dx/2, yi - dy/2,
                         xi + dx/2, yi + dy/2))

# Create GeoDataFrame
grid_1km = gpd.GeoDataFrame(geometry=polys, crs=HAD_Dolwen_ppn.rio.crs)

# Clip to catchment
grid_1km = gpd.overlay(grid_1km, catchment_Dolwen[["geometry"]], how="intersection")
grid_1km = grid_1km.reset_index(drop=True)

### Summarise slope/aspect/elevation inside each 1km grid cell

Could this just be done with regridding?

In [7]:
# Step 1: Give each pixel an ID
grid_1km["zone_id"] = np.arange(len(grid_1km))

# Step 2: Rasterise polygons, so each pixel stores its grid ID
zone_raster = rasterize(
    [(geom, value) for geom, value in zip(grid_1km.geometry, grid_1km.zone_id)],
    out_shape=dem_arr.shape,
    transform=transform,
    fill=-1,   # background
    dtype="int32")

# Step 3: Convert to xarray (so zone map is an xarray object, same shape as the DEM, slope and aspect)
zones = xr.DataArray(zone_raster, dims=("y", "x"))
dem_xr = xr.DataArray(dem_arr, dims=("y", "x"))
slope_xr = xr.DataArray(slope_arr, dims=("y", "x"))
aspect_xr = xr.DataArray(aspect_arr, dims=("y", "x"))

# Step 4: Mask out irrelevant pixels
mask = zones >= 0
zones = zones.where(mask)
dem_xr = dem_xr.where(mask)
slope_xr = slope_xr.where(mask)
aspect_xr = aspect_xr.where(mask)

# Group all DEM pixels by their zone_id
dem_stats = dem_xr.groupby(zones)

# Step 6: Compute stats
### Elevation
dem_df = xr.Dataset({"mean_dem": dem_stats.mean(), "min_dem": dem_stats.min(),"max_dem": dem_stats.max(),  
                     "std_dem": dem_stats.std()}).to_dataframe()

#### Slope
slope_df = xr.Dataset({
    "mean_slope": slope_xr.groupby(zones).mean(),
    "std_slope": slope_xr.groupby(zones).std()}).to_dataframe()

### Aspect
aspect_df = xr.Dataset({
    "mean_aspect": aspect_xr.groupby(zones).mean(),
    "std_aspect": aspect_xr.groupby(zones).std()}).to_dataframe()


### Contain one dataset which includes all of the variables for each 1km grid cell within the catchment

In [8]:
grid_1km = (grid_1km
        .merge(dem_df, left_index=True, right_index=True)
          .merge(aspect_df, left_index=True, right_index=True)
          .merge(slope_df,  left_index=True, right_index=True))

grid_1km["aspect_sin"] = np.sin(np.deg2rad(grid_1km["mean_aspect"]))
grid_1km["aspect_cos"] = np.cos(np.deg2rad(grid_1km["mean_aspect"]))

cell_centroids = grid_1km.geometry.centroid

grid_1km["x_coord"] = cell_centroids.x
grid_1km["y_coord"] = cell_centroids.y

### Keep only cells with a substantial portion of cell within catchment

In [9]:
grid_1km["cell_area"] = grid_1km.geometry.area

intersection = gpd.overlay(grid_1km,catchment_Dolwen[["geometry"]],how="intersection")

intersection["intersect_area"] = intersection.geometry.area

grid_1km = grid_1km.merge(intersection.groupby(intersection.index)["intersect_area"].sum(),left_index=True,right_index=True,
                            how="left")

grid_1km["intersect_area"] = grid_1km["intersect_area"].fillna(0)

grid_1km["coverage"] = grid_1km["intersect_area"] / grid_1km["cell_area"]

grid_1km = grid_1km[grid_1km["coverage"] >= 0.5]
grid_1km = grid_1km[grid_1km['intersect_area']>100000].copy()
len(grid_1km)

208

<div style="background-color: lightblue; color: black;  border: 2px solid black; padding: 15px 20px; border-radius: 5px; margin: 10px 0px;"><h3 style="color: black; margin: 0px;"> Implement a cluster analysis </h3>
    
This is designed to find the best way to partition the 1km grid cells into groups which have similar environmental characteristics.    
    
 - Define the optimal number of clusters using silhouette scores
 - Run the cluster analysis
    
  </div>    
</div>

In [10]:
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = grid_1km[["mean_dem", "mean_slope","aspect_sin","aspect_cos",]]# "x_coord", "y_coord"]]
scaler = StandardScaler()
X = scaler.fit_transform(features)

for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=0)
    labels = km.fit_predict(X)
    print(k, silhouette_score(X, labels))

2 0.27454230693778914
3 0.279364184207456
4 0.24562322520958843
5 0.22429993700617012
6 0.2233826671900756
7 0.22720506990873343
8 0.23923432382803672
9 0.2386470912702017


#### Run the cluster analysis

In [11]:
k = 7  # start here
kmeans = KMeans(n_clusters=k, random_state=0)
labels = kmeans.fit_predict(X)

# Attach labels back to data
grid_1km["cluster"] = labels

<div style="background-color: lightblue; color: black;  border: 2px solid black; padding: 15px 20px; border-radius: 5px; margin: 10px 0px;"><h3 style="color: black; margin: 0px;"> Decide on one grid cell to represent each cluster </h3>
    
    
  </div>    
</div>

#### Find one cell to represent each cluster

In [12]:
# selected_cells = []

# for c in np.unique(labels):
#     cluster_idx = np.where(labels == c)[0]
#     cluster_points = X[cluster_idx]
    
#     centroid = kmeans.cluster_centers_[c]
    
#     # find closest real point
#     distances = np.linalg.norm(cluster_points - centroid, axis=1)
#     best_idx = cluster_idx[np.argmin(distances)]
    
#     selected_cells.append(best_idx)
    
# gauge_locations = grid_1km.iloc[selected_cells]    
# selected_cells

#### Find one cell to represent each cluster, with constraint on proximity to each other

In [13]:
existing_gauges = rain_meta_sf.to_crs(grid_1km.crs)
existing_coords = np.array([(g.x, g.y) for g in existing_gauges.geometry])

In [14]:
rain_meta_sf = rain_meta_sf.to_crs(grid_1km.crs)
existing_coords = np.array([(geom.x, geom.y) for geom in rain_meta_sf.geometry])

cell_xy = np.array([(geom.x, geom.y) for geom in cell_centroids])

candidate_xy = cell_xy[idx]

too_close = False

# 1. check against already selected new gauges
if len(selected_cells) > 0:
    d_new = np.linalg.norm(cell_xy[selected_cells] - candidate_xy, axis=1)
    if np.any(d_new < min_dist):
        too_close = True

# 2. check against existing gauges
if not too_close and len(existing_coords) > 0:
    d_exist = np.linalg.norm(existing_coords - candidate_xy,axis=1)
    if np.any(d_exist < min_dist):
        too_close = True
        
def violates_constraints(point, selected_xy, existing_xy, min_dist):
    if len(selected_xy) > 0:
        if np.any(np.linalg.norm(selected_xy - point, axis=1) < min_dist):
            return True
    if len(existing_xy) > 0:
        if np.any(np.linalg.norm(existing_xy - point, axis=1) < min_dist):
            return True
    return False        

cell_xy = np.array([
    (geom.x, geom.y) for geom in cell_centroids
])

existing_xy = np.array([
    (geom.x, geom.y) for geom in rain_meta_sf.geometry
])

selected_cells = []

min_dist = 5000

for c in cluster_candidates:
    
    candidates = cluster_candidates[c]
    
    chosen = None
    
    for idx in candidates:
        
        candidate_xy = cell_xy[idx]
        
        # build current selected coordinates
        selected_xy = cell_xy[selected_cells] if len(selected_cells) > 0 else np.empty((0, 2))
        
        # check constraints
        if not violates_constraints(candidate_xy, selected_xy, existing_xy, min_dist):
            chosen = idx
            break   # stop at first valid candidate
    
    # fallback if nothing satisfies constraints
#     if chosen is None:
#         print(c, "Fallback used")
#         chosen = candidates[0]
    
    selected_cells.append(chosen)

    
gauge_locations = grid_1km.iloc[selected_cells].copy()
gauge_points = gauge_locations.copy()
gauge_points["geometry"] = gauge_points.geometry.centroid

fig, ax = plt.subplots(figsize=(8, 8))
grid_1km.boundary.plot(ax=ax, color="white", linewidth=1) # Grid outlines
grid_1km.plot(column="cluster",categorical=True,legend=True,ax=ax,alpha=1 , cmap='Blues')
grid_1km.boundary.plot(ax=ax, color="white", linewidth=1) # Grid outlines

catchment_Dolwen.boundary.plot(ax=ax, color="black")
gauge_points.plot(ax=ax,color="black",markersize=30,label="Selected gauge locations")
rain_meta_sf.plot(ax=ax, color="red", markersize=30, zorder=5)

for pt in gauge_points.geometry:
    circle = plt.Circle((pt.x, pt.y), min_dist/2, color='red', fill=False, linestyle='--')
    ax.add_patch(circle)
    
# After selection, verify no two selected cells are too close
coords = np.array([(cell_centroids.iloc[i].x, cell_centroids.iloc[i].y) 
                   for i in selected_cells])
dist_matrix = cdist(coords, coords)
np.fill_diagonal(dist_matrix, np.inf)  # ignore self-distances
print("Minimum distance between selected cells:", dist_matrix.min())
print("Any violations?", (dist_matrix < min_dist).any())      

NameError: name 'idx' is not defined

In [ ]:
cell_xy = np.array([
    (geom.x, geom.y) for geom in cell_centroids])

existing_xy = np.array([
    (geom.x, geom.y) for geom in rain_meta_sf.geometry])

selected_cells = []
selected_xy = existing_xy.copy()

min_dist = 5000

selected_cells = []

for c in cluster_candidates:
    
    candidates = cluster_candidates[c]
    
    best_idx = None
    best_score = -np.inf
    
    for idx in candidates:
        
        candidate_xy = cell_xy[idx]
        
        # distance to ALL already selected (including existing)
        if len(selected_xy) == 0:
            min_dist = np.inf
        else:
            dists = np.linalg.norm(selected_xy - candidate_xy, axis=1)
            min_dist = np.min(dists)
        
        # MAXIMISE this minimum distance
        if min_dist > best_score:
            best_score = min_dist
            best_idx = idx
    
    selected_cells.append(best_idx)
    
    # update selected set
    selected_xy = np.vstack([selected_xy, cell_xy[best_idx]])
    
gauge_locations = grid_1km.iloc[selected_cells].copy()
gauge_points = gauge_locations.copy()
gauge_points["geometry"] = gauge_points.geometry.centroid

fig, ax = plt.subplots(figsize=(8, 8))
grid_1km.boundary.plot(ax=ax, color="white", linewidth=1) # Grid outlines
grid_1km.plot(column="cluster",categorical=True,legend=True,ax=ax,alpha=1 , cmap='Blues')
grid_1km.boundary.plot(ax=ax, color="white", linewidth=1) # Grid outlines

catchment_Dolwen.boundary.plot(ax=ax, color="black")
gauge_points.plot(ax=ax,color="black",markersize=30,label="Selected gauge locations")
rain_meta_sf.plot(ax=ax, color="red", markersize=30, zorder=5)

min_dist= 5000
for pt in gauge_points.geometry:
    circle = plt.Circle((pt.x, pt.y), min_dist/2, color='red', fill=False, linestyle='--')
    ax.add_patch(circle)
    
# After selection, verify no two selected cells are too close
coords = np.array([(cell_centroids.iloc[i].x, cell_centroids.iloc[i].y) 
                   for i in selected_cells])
dist_matrix = cdist(coords, coords)
np.fill_diagonal(dist_matrix, np.inf)  # ignore self-distances
print("Minimum distance between selected cells:", dist_matrix.min())
print("Any violations?", (dist_matrix < min_dist).any())          

In [ ]:
cell_xy = np.array([
    (geom.x, geom.y) for geom in cell_centroids])

existing_xy = np.array([
    (geom.x, geom.y) for geom in rain_meta_sf.geometry])

def violates_hard_constraint(candidate_xy, selected_xy, existing_xy, min_dist):

    # check selected gauges
    if len(selected_xy) > 0:
        d_new = np.linalg.norm(selected_xy - candidate_xy, axis=1)
        if np.any(d_new < min_dist):
            return True

    # check existing gauges
    if len(existing_xy) > 0:
        d_exist = np.linalg.norm(existing_xy - candidate_xy, axis=1)
        if np.any(d_exist < min_dist):
            return True

    return False


selected_cells = []
selected_xy = np.empty((0, 2))

min_dist = 4000  # hard constraint

for c in cluster_candidates:

    candidates = cluster_candidates[c]

    best_idx = None
    best_score = -np.inf

    for idx in candidates:

        candidate_xy = cell_xy[idx]

        # ❌ hard constraint check
        if violates_hard_constraint(candidate_xy, selected_xy, existing_xy, min_dist):
            continue

        # 🎯 maximin score (distance to nearest selected/existing)
        if len(selected_xy) == 0 and len(existing_xy) == 0:
            score = np.inf
        else:
            all_ref = np.vstack([selected_xy, existing_xy]) if len(selected_xy) > 0 else existing_xy
            dists = np.linalg.norm(all_ref - candidate_xy, axis=1)
            score = np.min(dists)

        # keep best
        if score > best_score:
            best_score = score
            best_idx = idx

    # fallback (if nothing satisfies constraint)
    if best_idx is None:
        print(f"fallback used for cluster {c}")
        best_idx = candidates[0]

    selected_cells.append(best_idx)

    # update selected set
    selected_xy = np.vstack([selected_xy, cell_xy[best_idx]])
    
    
gauge_locations = grid_1km.iloc[selected_cells].copy()
gauge_points = gauge_locations.copy()
gauge_points["geometry"] = gauge_points.geometry.centroid

fig, ax = plt.subplots(figsize=(8, 8))
grid_1km.plot(column="cluster",categorical=True,legend=True,ax=ax,alpha=1 , cmap='Blues')
grid_1km.boundary.plot(ax=ax, color="white", linewidth=1) # Grid outlines
catchment_Dolwen.boundary.plot(ax=ax, color="black")
gauge_points.plot(ax=ax,color="black",markersize=30,label="Selected gauge locations")
rain_meta_sf.plot(ax=ax, color="red", markersize=30, zorder=5)

for pt in gauge_points.geometry:
    circle = plt.Circle((pt.x, pt.y), min_dist/2, color='red', fill=False, linestyle='--')
    ax.add_patch(circle)
    
# After selection, verify no two selected cells are too close
coords = np.array([(cell_centroids.iloc[i].x, cell_centroids.iloc[i].y) 
                   for i in selected_cells])
dist_matrix = cdist(coords, coords)
np.fill_diagonal(dist_matrix, np.inf)  # ignore self-distances
print("Minimum distance between selected cells:", dist_matrix.min())
print("Any violations?", (dist_matrix < min_dist).any())        

In [ ]:
# import geopandas as gpd

# grid_1km["cell_area"] = grid_1km.geometry.area
# intersection = gpd.overlay(
#     grid_1km,
#     catchment_Dolwen[["geometry"]],
#     how="intersection"
# )

# intersection["intersect_area"] = intersection.geometry.area

# coverage = intersection.groupby(intersection.index)["intersect_area"].sum()
# grid_1km["intersect_area"] = coverage
# grid_1km["intersect_area"] = grid_1km["intersect_area"].fillna(0)

# grid_1km["coverage"] = grid_1km["intersect_area"] / grid_1km["cell_area"]
# grid_1km

In [ ]:
selected_cells

In [ ]:
from scipy.spatial.distance import cdist

# ------------------------------------ #
# For each of the clusters, find a list of the cells which 
# would best represent the cluster, in order of appropriateness
# This is based on: how close each grid cell is to the cluster centroid (in feature space)
# ------------------------------------ #
cluster_candidates = {}

# Loop through each grid cell (and the cluster label assigned to it)
for c in np.unique(labels):
    cluster_idx = np.where(labels == c)[0]
    cluster_points = X[cluster_idx]
    
    centroid = kmeans.cluster_centers_[c]
    
    # distances to centroid
    distances = np.linalg.norm(cluster_points - centroid, axis=1)
    
    # sort candidates (best first)
    sorted_idx = cluster_idx[np.argsort(distances)]
    
    cluster_candidates[c] = sorted_idx

# ------------------------------------ #
# This code then loops through the candidate locations in order from best -> worst
# For each candidate:
#    - check distance to selected gauges
#    - check distance to existing gauges
#    - if valid → accept and STOP
# Checks if the top ranked cell is closer than the min_dist threshold to any of the other
# cells selected to represent the other clusters
# ------------------------------------ #    
    
min_dist = 3000
selected_cells = []

for c in cluster_candidates:
    candidates = cluster_candidates[c]
    
    chosen = None
    
    for idx in candidates:
        candidate_point = cell_centroids.iloc[idx]
        too_close = False

        # 1. check against selected gauges
        for sel in selected_cells:
            dist = candidate_point.distance(cell_centroids.iloc[sel])
            if dist < min_dist:
                too_close = True
                break

        # 2. check against existing gauges
        if not too_close:
            for ex in existing_coords:
                dist = np.linalg.norm(
                    np.array([candidate_point.x, candidate_point.y]) - ex
                )
                if dist < min_dist:
                    too_close = True
                    break

        # ✅ only accept if passes BOTH checks
        if not too_close:
            chosen = idx
            break

    # ⚠️ safer fallback
    if chosen is None:
        print(f"WARNING: No valid candidate for cluster {c}")
        continue   # ← better than forcing invalid choice
#         chosen = candidates[0]

    selected_cells.append(chosen)   

gauge_locations = grid_1km.iloc[selected_cells].copy()
gauge_points = gauge_locations.copy()
gauge_points["geometry"] = gauge_points.geometry.centroid

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_aspect('equal')
grid_1km.plot(column="cluster",categorical=True,legend=True,ax=ax,alpha=1 , cmap='Blues')
grid_1km.boundary.plot(ax=ax, color="white", linewidth=1) # Grid outlines
catchment_Dolwen.boundary.plot(ax=ax, color="black")
gauge_points.plot(ax=ax,color="black",markersize=30,label="Selected gauge locations")
rain_meta_sf.plot(ax=ax, color="red", markersize=50, marker = "X", zorder=5)

for ex in existing_coords:
    circle = plt.Circle((ex[0], ex[1]), min_dist/2,
                        color='blue', fill=False, linestyle='--')
    ax.add_patch(circle)

for pt in gauge_points.geometry:
    circle = plt.Circle((pt.x, pt.y), min_dist/2, color='red', fill=False, linestyle='--')
    ax.add_patch(circle)
    
# After selection, verify no two selected cells are too close
coords = np.array([(cell_centroids.iloc[i].x, cell_centroids.iloc[i].y) 
                   for i in selected_cells])
dist_matrix = cdist(coords, coords)
np.fill_diagonal(dist_matrix, np.inf)  # ignore self-distances
print("Minimum distance between selected cells:", dist_matrix.min())
print("Any violations?", (dist_matrix < min_dist).any())   

# check against existing gauges
for i, coord in enumerate(coords):
    for ex in existing_coords:
        dist = np.linalg.norm(coord - ex)
        if dist < min_dist:
            print(f"Gauge {i} too close to existing gauge")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
# grid_1km.plot(column="cluster",categorical=True,legend=True,ax=ax,alpha=1 , cmap='viridis')
grid_1km.plot(column="cluster",categorical=True,legend=True,ax=ax,alpha=1 , cmap='Blues')
grid_1km.boundary.plot(ax=ax, color="white", linewidth=1) # Grid outlines
catchment_Dolwen.boundary.plot(ax=ax, color="black")
gauge_points.plot(ax=ax,color="black", markersize=30,label="Selected gauge locations")
rain_meta_sf.plot(ax=ax, color="red", markersize=50, marker = "X", zorder=5)

for pt in gauge_points.geometry:
    circle = plt.Circle((pt.x, pt.y), min_dist/2, color='red', fill=False, linestyle='--')
    ax.add_patch(circle)

In [ ]:
# coords = np.column_stack([centroids.x, centroids.y])  # shape (n_cells, 2)

# selected_cells = []

# clusters = np.unique(labels)

# # 1. Initialise with one cluster (e.g. first)
# c0 = clusters[0]
# cluster_idx = np.where(labels == c0)[0]
# selected_cells.append(cluster_idx[0])  # or best representative

# # 2. Iterate over remaining clusters
# for c in clusters[1:]:
#     cluster_idx = np.where(labels == c)[0]
    
#     best_idx = None
#     best_score = -np.inf
    
#     for idx in cluster_idx:
#         point = coords[idx]
        
#         # distance to closest selected point
#         dists = [
#             np.linalg.norm(point - coords[s])
#             for s in selected_cells
#         ]
        
#         score = min(dists)  # maximise minimum distance
        
#         if score > best_score:
#             best_score = score
#             best_idx = idx
    
#     selected_cells.append(best_idx)

# gauge_locations = grid_1km.iloc[selected_cells]


In [ ]:
# import seaborn as sns
# sns.pairplot(gauge_locations, vars=["mean_dem", "mean_slope", 'mean_aspect'], hue="cluster", palette = 'viridis')

In [ ]:
# import rasterstats as rs

# rain_stats = rs.zonal_stats(
#     grid_1km,
#     HAD_Dolwen_ppn.mean(dim="time").values,
#     affine=HAD_Dolwen_ppn.rio.transform(),
#     stats="mean",
#     nodata=np.nan
# )

# grid_1km["rain"] = [r["mean"] for r in rain_stats]

In [ ]:
# import numpy as np
# from scipy.spatial.distance import pdist

# # 1. Get centroids
# centroids = grid_1km.geometry.centroid
# coords = np.array([(pt.x, pt.y) for pt in centroids])

# # 2. Get rainfall values
# rain = HAD_Dolwen_ppn.mean(dim="time").values.flatten()
# rain = rain[~np.isnan(rain)]

# # 3. Mask NaNs (CRITICAL)
# mask = ~np.isnan(rain)

# coords = coords[mask]
# values = rain[mask]

# # 4. Now compute distances + differences
# dists = pdist(coords)
# diffs = pdist(values.reshape(-1, 1))

# gamma = 0.5 * (diffs ** 2)

In [ ]:
# bins = np.linspace(0, dists.max(), 20)
# bin_idx = np.digitize(dists, bins)
# # 
# variogram = [gamma[bin_idx == i].mean() for i in range(1, len(bins))]


In [ ]:
# import matplotlib.pyplot as plt

# plt.plot(bins[1:], variogram, 'o-')
# plt.xlabel("Distance (m)")
# plt.ylabel("Semivariance")
# plt.title("Empirical Variogram")
# plt.show()

In [ ]:
# grid_1km_sf["max_elev"]   = [z["max"]  for z in dem_stats]
# grid_1km_sf["min_elev"]   = [z["min"]  for z in dem_stats]
# grid_1km_sf["mean_elev"]  = [z["mean"] for z in dem_stats]
# grid_1km_sf["var_elev"]   = [z["std"]**2 if z["std"] is not None else np.nan for z in dem_stats]
# grid_1km_sf["range_elev"] = grid_1km_sf["max_elev"] - grid_1km_sf["min_elev"]

# grid_1km_sf["mean_slope"] = [z["mean"] for z in slope_stats]
# grid_1km_sf["var_slope"]  = [z["std"]**2 if z["std"] is not None else np.nan for z in slope_stats]

# grid_1km_sf["mean_aspect"]= [z["mean"] for z in aspect_stats]
# grid_1km_sf["var_aspect"] = [z["std"]**2 if z["std"] is not None else np.nan for z in aspect_stats]

# grid_1km_sf["mean_tri"]   = [z["mean"] for z in tri_stats]
# grid_1km_sf["var_tri"]    = [z["std"]**2 if z["std"] is not None else np.nan for z in tri_stats]

# grid_1km_sf["mean_PPN"]   = [z["mean"] for z in ppn_stats]

# # ==============================================================
# Point grid and rain gauge spatial joins
# ==============================================================

# Intersect grid polygons with centroid points → keep terrain attributes per point
# point_1km_sf = gpd.sjoin(grid_1km_sf, grid_1km_point, how="inner", predicate="contains")
# point_1km_sf["x"] = grid_1km_xy[:, 0]
# point_1km_sf["y"] = grid_1km_xy[:, 1]

# # Join rain gauge metadata with grid cell attributes
# rain_meta_sf2 = gpd.sjoin(rain_meta_sf, grid_1km_sf, how="left", predicate="within")
# rain_meta_sf2 = rain_meta_sf2.iloc[:, :12]  # keep first 12 columns (matching R)

# rain_xy = np.array([(g.x, g.y) for g in rain_meta_sf2.geometry])
# rain_meta_sf2["x"] = rain_xy[:, 0]
# rain_meta_sf2["y"] = rain_xy[:, 1]

In [ ]:
HAD_Dolwen_ppn_mean
values = HAD_Dolwen_ppn.mean(dim="time").values.flatten()
values.shape

In [ ]:
# ==============================================================
# Plot: rain gauges, river network, mean rainfall, grid
# ==============================================================

fig, ax = plt.subplots(figsize=(10, 10))

# Mean rainfall raster
HAD_Dolwen_ppn_mean.plot( ax=ax,cmap="magma", alpha=0.8,add_colorbar=True,robust=True)

# Grid outlines
grid_1km.boundary.plot(ax=ax, color="grey", linewidth=1)

# River network
# river_net_Dolwen.plot(ax=ax, color="steelblue", linewidth=0.8)

# Rain gauges
rain_meta_sf.plot(ax=ax, color="red", markersize=30, zorder=5)

plt.title("Mean Annual Rainfall with Rain Gauges and Grid")
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================
# Create 1km polygon grid and centroid points
# ==============================================================

# Build 1km grid polygons clipped to catchment
from shapely.geometry import box

# Get raster bounds and build grid
xmin, ymin, xmax, ymax = catchment_Dolwen.total_bounds
cell_size = 1000

cols = np.arange(xmin, xmax, cell_size)
rows = np.arange(ymin, ymax, cell_size)

grid_polys = [box(x, y, x + cell_size, y + cell_size)
    for x in cols for y in rows]

grid_1km = gpd.GeoDataFrame(geometry=grid_polys, crs=ukgrid)

# Clip grid to catchment boundary
grid_1km = gpd.overlay(grid_1km, catchment_Dolwen[["geometry"]], how="intersection")
grid_1km = grid_1km.reset_index(drop=True)

# Get centroids
grid_1km_point = grid_1km.copy()
grid_1km_point["geometry"] = grid_1km.centroid

# Extract XY coordinates
grid_1km_xy = np.array(
    [(geom.x, geom.y) for geom in grid_1km_point.geometry])

# Quick plot to check
plt.scatter(grid_1km_xy[:, 0], grid_1km_xy[:, 1], s=2)
plt.title("1km Grid Centroids")
plt.axis("equal")
plt.show()